<a href="https://colab.research.google.com/github/Brokenstarck/Optativa-ia/blob/main/pr%C3%A1ctica_2_3_teachable_machine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Teachable Machine**

En esta práctica se debe exportar el modelo que se genera en Teachable Machine.

A la hora de exportar el modelo, se puede exportar para Tensorflow.

También viene un fragmento de código para usar el modelo:

```
from keras.models import load_model  # TensorFlow is required for Keras to work
from PIL import Image, ImageOps  # Install pillow instead of PIL
import numpy as np

# Disable scientific notation for clarity
np.set_printoptions(suppress=True)

# Load the model
model = load_model("keras_Model.h5", compile=False)

# Load the labels
class_names = open("labels.txt", "r").readlines()

# Create the array of the right shape to feed into the keras model
# The 'length' or number of images you can put into the array is
# determined by the first position in the shape tuple, in this case 1
data = np.ndarray(shape=(1, 224, 224, 3), dtype=np.float32)

# Replace this with the path to your image
image = Image.open("<IMAGE_PATH>").convert("RGB")

# resizing the image to be at least 224x224 and then cropping from the center
size = (224, 224)
image = ImageOps.fit(image, size, Image.Resampling.LANCZOS)

# turn the image into a numpy array
image_array = np.asarray(image)

# Normalize the image
normalized_image_array = (image_array.astype(np.float32) / 127.5) - 1

# Load the image into the array
data[0] = normalized_image_array

# Predicts the model
prediction = model.predict(data)
index = np.argmax(prediction)
class_name = class_names[index]
confidence_score = prediction[0][index]

# Print prediction and confidence score
print("Class:", class_name[2:], end="")
print("Confidence Score:", confidence_score)

```

**Tarea**: subir 3 imágenes propias (una clara, una ambigua, una que no pertenezca a ninguna categoría) y anotar qué predice y con qué confianza.

**¡Ojo! Hay que cargar una  versión anterior de Keras**

In [4]:
# forzar TensorFlow a usar el comportamiento clásico de Keras 2
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import tensorflow as tf
import tf_keras as keras

print("Versión de TensorFlow:", tf.__version__)
print("Versión de TF-Keras (Legacy):", keras.__version__)

try:
    # Cargamos el modelo utilizando el módulo de compatibilidad oficial tf_keras
    model = keras.models.load_model('keras_model.h5', compile=False)
    print("\n¡MODELO CARGADO CON ÉXITO!")
    model.summary()
except Exception as e:
    print(f"\nError al cargar: {e}")

Versión de TensorFlow: 2.21.0
Versión de TF-Keras (Legacy): 2.21.0

¡MODELO CARGADO CON ÉXITO!
Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sequential_1 (Sequential)   (None, 1280)              410208    
                                                                 
 sequential_3 (Sequential)   (None, 2)                 128300    
                                                                 
Total params: 538508 (2.05 MB)
Trainable params: 524428 (2.00 MB)
Non-trainable params: 14080 (55.00 KB)
_________________________________________________________________


In [8]:


from PIL import Image, ImageOps
import numpy as np

# Cargar las etiquetas
class_names = open("labels.txt", "r").readlines()

# Función para analizar imágenes
def analizar_imagen(ruta):

    data = np.ndarray(shape=(1, 224, 224, 3), dtype=np.float32)

    image = Image.open(ruta).convert("RGB")
    image = ImageOps.fit(image, (224, 224), Image.Resampling.LANCZOS)

    image_array = np.asarray(image)
    normalized_image_array = (image_array.astype(np.float32) / 127.5) - 1

    data[0] = normalized_image_array

    prediction = model.predict(data, verbose=0)

    index = np.argmax(prediction)
    class_name = class_names[index]
    confidence_score = prediction[0][index]

    print("\nImagen:", ruta)
    print("Clase:", class_name[2:].strip())
    print("Confianza:", round(float(confidence_score) * 100, 2), "%")







Imagen: 11.png
Clase: Taza
Confianza: 100.0 %

Imagen: 22.png
Clase: Botella
Confianza: 99.98 %

Imagen: 33.png
Clase: Taza
Confianza: 83.84 %


In [9]:
# implementa la clasificación con la imagen que pertenece claramente a una categoría
analizar_imagen("11.png")


Imagen: 11.png
Clase: Taza
Confianza: 100.0 %


In [10]:
# implementa la clasificación con una imagen ambigua, que puede ser de cualquier categoría
analizar_imagen("22.png")


Imagen: 22.png
Clase: Botella
Confianza: 99.98 %


In [11]:
# implementa la clasificación con una imagen sin ninguna categoría del modelo
analizar_imagen("33.png")


Imagen: 33.png
Clase: Taza
Confianza: 83.84 %


**¿Cómo se usa el modelo exportado? ¿Cómo se comporta?**

---



El modelo se utiliza con Python, cargando el archivo entrenado y las etiquetas. Después se introduce una imagen, se adapta a 224x224 píxeles y se analiza mediante `model.predict()`.

El modelo reconoce si una imagen es una taza o una botella y muestra su porcentaje de confianza. Con imágenes claras suele acertar, pero puede confundirse con imágenes ambiguas o desconocidas, ya que siempre intenta clasificarlas en una de las dos categorías.